In [1]:
!pip install -q sentence-transformers faiss-cpu transformers pandas

In [2]:
#Load E-commerce Policy Documents
documents = [
    "Return Policy: Customers can return any item within 30 days of purchase. The item must be unused and in its original packaging. A receipt is required for all returns.",
    "Refund Policy: Once a return is received, refunds will be processed within 7 business days to the original method of payment.",
    "Shipping Policy: Standard shipping takes 3 to 5 business days. Express shipping takes 1 to 2 business days. Free shipping is available on orders over $50.",
    "Order Tracking: Once your order has shipped, you will receive an email with a tracking number. You can track your order on our website.",
    "Payment Methods: We accept Visa, MasterCard, American Express, and PayPal. Cash on Delivery (COD) is only available for orders under $100.",
    "Warranty: All electronics come with a 1-year manufacturer warranty. The warranty does not cover physical damage."
]

In [3]:
#Split text into smaller chunks
def split_text(text, chunk_size=150, overlap=20):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks

all_chunks = []
for doc in documents:
    all_chunks.extend(split_text(doc))

print(f" Total chunks created: {len(all_chunks)}")
print("Example Chunk:\n", all_chunks[0])

 Total chunks created: 10
Example Chunk:
 Return Policy: Customers can return any item within 30 days of purchase. The item must be unused and in its original packaging. A receipt is required 


In [4]:
#Create Embeddings and FAISS Index
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading embedding model (all-MiniLM-L6-v2)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Creating embeddings...")
embeddings = embedder.encode(all_chunks)
embeddings = np.array(embeddings).astype('float32')

print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print(f" FAISS Database ready with {index.ntotal} vectors!")

Loading embedding model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...
Building FAISS index...
 FAISS Database ready with 10 vectors!


In [5]:
#Load FLAN-T5 directly without pipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Loading FLAN-T5-small model...")
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def ask_rag_bot(question):
    # 1. Convert question to embedding
    question_embedding = embedder.encode([question]).astype('float32')

    # 2. Search FAISS for top 3 relevant chunks
    distances, indices = index.search(question_embedding, k=3)

    # 3. Combine retrieved chunks as context
    context = "\n".join([all_chunks[i] for i in indices[0]])

    # 4. Create prompt
    prompt = f"Answer the question using only the context below. If the answer is not in the context, say you do not know.\n\nContext: {context}\n\nQuestion: {question}"

    # 5. Generate answer using the model directly
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_length=100, do_sample=False)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

print(" Chatbot is Ready! Testing with a sample question...\n")

# Test the bot
test_question = "How many days do I have to return an item?"
answer = ask_rag_bot(test_question)

print(f"User: {test_question}")
print(f"Bot: {answer}")

Loading FLAN-T5-small model...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

 Chatbot is Ready! Testing with a sample question...

User: How many days do I have to return an item?
Bot: 30


In [6]:
#Test the bot with custom questions
my_question = "What payment methods do you accept?"
print(f"User: {my_question}")
print(f"Bot: {ask_rag_bot(my_question)}")

print("\n-------------------\n")

my_question_2 = "How can I track my order?"
print(f"User: {my_question_2}")
print(f"Bot: {ask_rag_bot(my_question_2)}")

print("\n-------------------\n")

my_question_3 = "How long does a refund take?"
print(f"User: {my_question_3}")
print(f"Bot: {ask_rag_bot(my_question_3)}")

User: What payment methods do you accept?
Bot: Visa, MasterCard, American Express, and PayPal

-------------------

User: How can I track my order?
Bot: You can track your order on our website. able on orders over $50

-------------------

User: How long does a refund take?
Bot: 7 business days


In [7]:
#Evaluation (Response Time & Testing)
import time

test_questions = [
    "What is the return window?",
    "How long does standard shipping take?",
    "Do you accept PayPal?",
    "What is the warranty on electronics?",
    "Can I return an item after 30 days?"
]

print("--- Evaluation Results ---")
total_time = 0

for q in test_questions:
    start_time = time.time()
    ans = ask_rag_bot(q)
    end_time = time.time()

    time_taken = end_time - start_time
    total_time += time_taken

    print(f"\nQ: {q}")
    print(f"A: {ans}")
    print(f"Time taken: {time_taken:.2f} seconds")

print(f"\n Average Response Time: {total_time/len(test_questions):.2f} seconds per query.")

--- Evaluation Results ---

Q: What is the return window?
A: Return Policy
Time taken: 0.32 seconds

Q: How long does standard shipping take?
A: 3 to 5 business days
Time taken: 0.54 seconds

Q: Do you accept PayPal?
A: Yes
Time taken: 0.30 seconds

Q: What is the warranty on electronics?
A: 1-year manufacturer warranty
Time taken: 0.39 seconds

Q: Can I return an item after 30 days?
A: Yes
Time taken: 0.33 seconds

 Average Response Time: 0.38 seconds per query.


In [8]:
# Gradio Web Interface for Live Demo
!pip install -q gradio
import gradio as gr

def chat_interface(question):
    if not question.strip():
        return "Please type a valid question."
    return ask_rag_bot(question)

# Create the Web App
demo = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(label="Customer Question", placeholder="e.g., What is the return policy? / How long is shipping?"),
    outputs=gr.Textbox(label="Bot Answer"),
    title="🛒 E-commerce Customer Support Bot (RAG)",
    description="Ask questions about returns, shipping, payments, or warranty."
)

# Launch the app with a public link
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5491768c30ee2d8805.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
